# Задание 4 

## Seed-and-Extend

Seed-and-Extend — это эвристический алгоритм поиска локального выравнивания последовательностей.  
Он является ускоренной альтернативой алгоритму Смита–Уотермана и позволяет избежать полного
вычисления матрицы размером O(n × m), что значительно ускоряет поиск на практике.

Алгоритм состоит из двух фаз:

### 1. Seeding (поиск семян)
- База данных разбивается на k-меры длины k.
- Для каждого k-мера создаётся индекс: словарь вида  
  k-mer → [список позиций в базе].
- Запрос также разбивается на все возможные k-меры.
- Для каждого k-мера запроса находятся точные совпадения в базе.
- Каждое совпадение образует seed — пару координат (i, j).

### 2. Extension (расширение)
- От каждого seed запускается расширение влево и вправо.
- Начальный счёт равен весу seed (идеальное совпадение).
- При каждом шаге вычисляется новый score:
  - match → +1  
  - mismatch → −1  
  - gap → −1  
- Поддерживаются два значения:
  - Scur — текущий счёт
  - Smax — максимальный достигнутый счёт
- Если разница Smax − Scur ≥ X (порог X-drop), расширение останавливается.
- Итоговое выравнивание обрезается по позиции, где достигнут Smax.

## Практическое задание

Реализовать алгоритм Seed-and-Extend для последовательностей:

- Референс: `CTAGGATCCAGGCATACGA`
- Последовательность для поиска: `GGATCCATTCATTA`

Параметры:
- k = 4
- X-drop = 2
- match = +1
- mismatch = −1
- gap = −1

В результате необходимо вывести:
- Индекс базы данных
- Все найденные seed
- Значения Scur при каждом расширении
- Итоговый Smax
- Последовательности после расширения
- Лучшее локальное выравнивание

In [2]:
import numpy as np

In [8]:
# Параметры алгоритма
k = 4
MATCH = 1
MISMATCH = -1
X_DROP = 2

reference = "CTAGGATCCAGGCATACGA"
query = "GGATCCATTCATTA"

In [9]:
# Построение индекса базы данных
def build_index(sequence, k):
    index = {}
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]
        if kmer in index:
            index[kmer].append(i)
        else:
            index[kmer] = [i]
    return index

In [10]:
# Построение индекса базы данных
def build_index(sequence, k):
    index = {}
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]
        if kmer in index:
            index[kmer].append(i)
        else:
            index[kmer] = [i]
    return index

In [11]:
# Поиск seed
def find_seeds(query, index, k):
    seeds = []
    for i in range(len(query) - k + 1):
        kmer = query[i:i+k]
        if kmer in index:
            for j in index[kmer]:
                seeds.append((i, j, kmer))
    return seeds

In [17]:
def extend_seed(q, d, qi, di, k):
    print(f"\nРасширяем seed '{q[qi:qi+k]}' в позициях Q:{qi}, D:{di}")

    Scur = k * MATCH
    Smax = Scur

    best_left = qi
    best_right = qi + k

    # Списки для отслеживания Scur и Smax
    scur_left_list = []
    smax_left_list = []

    # Расширение влево
    left_q = qi - 1
    left_d = di - 1

    while left_q >= 0 and left_d >= 0:
        if q[left_q] == d[left_d]:
            Scur += MATCH
        else:
            Scur += MISMATCH

        if Scur > Smax:
            Smax = Scur
            best_left = left_q

        scur_left_list.append(Scur)
        smax_left_list.append(Smax)

        if Smax - Scur >= X_DROP:
            break

        left_q -= 1
        left_d -= 1

    # Сохраняем левое расширение
    print("Левое расширение:")
    for i, (s_cur, s_max) in enumerate(zip(scur_left_list, smax_left_list), 1):
        print(f"Шаг {i}: Scur={s_cur}, Smax={s_max}")

    # Перед расширением вправо сбрасываем Scur к Smax
    Scur = Smax
    scur_right_list = []
    smax_right_list = []

    # Расширение вправо
    right_q = qi + k
    right_d = di + k

    while right_q < len(q) and right_d < len(d):
        if q[right_q] == d[right_d]:
            Scur += MATCH
        else:
            Scur += MISMATCH

        if Scur > Smax:
            Smax = Scur
            best_right = right_q + 1

        scur_right_list.append(Scur)
        smax_right_list.append(Smax)

        if Smax - Scur >= X_DROP:
            break

        right_q += 1
        right_d += 1

    # Сохраняем правое расширение
    print("Правое расширение:")
    for i, (s_cur, s_max) in enumerate(zip(scur_right_list, smax_right_list), 1):
        print(f"Шаг {i}: Scur={s_cur}, Smax={s_max}")

    # Итоговое выравнивание
    aligned_q = q[best_left:best_right]
    shift = qi - best_left
    aligned_d = d[di - shift: di - shift + len(aligned_q)]

    print(f"Итоговое выравнивание: {aligned_q}  ||  {aligned_d}")
    print(f"Итоговый Smax: {Smax}")
    print(f"Итоговый Scur: {Scur}\n")

    return Smax, aligned_q, aligned_d

In [19]:
index = build_index(reference, k)
seeds = find_seeds(query, index, k)

print("\nНайденные seed:")
for seed in seeds:
    print(seed)

best_score = float("-inf")
best_alignment = None

for qi, di, kmer in seeds:
    score, aq, ad = extend_seed(query, reference, qi, di, k)

    if score > best_score:
        best_score = score
        best_alignment = (kmer, score, aq, ad)

print("\nЛучшее выравнивание:")
print("Лучший k-мер:", best_alignment[0])
print("Максимальный score:", best_alignment[1])
print("Query_part:", best_alignment[2])
print("Ref_part:", best_alignment[3])


Найденные seed:
(0, 3, 'GGAT')
(1, 4, 'GATC')
(2, 5, 'ATCC')
(3, 6, 'TCCA')

Расширяем seed 'GGAT' в позициях Q:0, D:3
Левое расширение:
Правое расширение:
Шаг 1: Scur=5, Smax=5
Шаг 2: Scur=6, Smax=6
Шаг 3: Scur=7, Smax=7
Шаг 4: Scur=6, Smax=7
Шаг 5: Scur=5, Smax=7
Итоговое выравнивание: GGATCCA  ||  GGATCCA
Итоговый Smax: 7
Итоговый Scur: 5


Расширяем seed 'GATC' в позициях Q:1, D:4
Левое расширение:
Шаг 1: Scur=5, Smax=5
Правое расширение:
Шаг 1: Scur=6, Smax=6
Шаг 2: Scur=7, Smax=7
Шаг 3: Scur=6, Smax=7
Шаг 4: Scur=5, Smax=7
Итоговое выравнивание: GGATCCA  ||  GGATCCA
Итоговый Smax: 7
Итоговый Scur: 5


Расширяем seed 'ATCC' в позициях Q:2, D:5
Левое расширение:
Шаг 1: Scur=5, Smax=5
Шаг 2: Scur=6, Smax=6
Правое расширение:
Шаг 1: Scur=7, Smax=7
Шаг 2: Scur=6, Smax=7
Шаг 3: Scur=5, Smax=7
Итоговое выравнивание: GGATCCA  ||  GGATCCA
Итоговый Smax: 7
Итоговый Scur: 5


Расширяем seed 'TCCA' в позициях Q:3, D:6
Левое расширение:
Шаг 1: Scur=5, Smax=5
Шаг 2: Scur=6, Smax=6
Шаг 3: Scur

В итоге у нас получилось, что для **всех seed одинаковый Smax => все выравнивания здесь одинаково хороши** (поэтому код и выдал лучшим первое выранивание)